# 04. 컨텍스트 엔지니어링 & 메모리 심화 -- Static/Dynamic Context, InMemoryStore, Skills 패턴

LangGraph의 컨텍스트 시스템과 장기 메모리(Store)를 심층 학습합니다. 정적/동적 런타임 컨텍스트부터 시맨틱 검색 기반 장기 메모리, 그리고 Progressive Disclosure(Skills) 패턴까지 다룹니다.

## 학습 목표

- 컨텍스트 엔지니어링의 2차원(Mutability x Lifetime) 매트릭스를 이해한다
- `context_schema` + `@dataclass`로 정적 런타임 컨텍스트를 구현한다
- `state_schema`와 `AgentState` 커스텀으로 동적 런타임 컨텍스트를 관리한다
- `InMemoryStore`의 namespace, put, get, search API를 활용한다
- 시맨틱 검색 기반 장기 메모리를 구축한다
- 메모리 3유형(Semantic, Episodic, Procedural)을 구분하여 설계한다
- Skills 패턴으로 Progressive Disclosure를 구현한다
- Hot path vs Background 메모리 쓰기 전략을 비교한다

## 4.1 환경 설정

In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

model = ChatOpenAI(model="gpt-4.1")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("환경 준비 완료.")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 4.2 컨텍스트 엔지니어링 개요

컨텍스트 엔지니어링은 **"올바른 정보를, 올바른 형식으로, 올바른 시점에"** AI에 제공하는 시스템 설계입니다. 단순한 프롬프트 엔지니어링을 넘어, 컨텍스트를 **런타임에 프로그래밍 방식으로 조립**하는 아키텍처적 접근입니다.

에이전트가 실패하는 주된 원인은 두 가지입니다:
1. LLM 능력 부족
2. **컨텍스트 부족 또는 부적절한 컨텍스트** (더 빈번한 원인)

따라서 컨텍스트 엔지니어링은 AI 엔지니어의 핵심 역할이며, 에이전트 신뢰성의 근본적인 해결책입니다.

### 2차원 매트릭스: Mutability x Lifetime

|  | **Runtime** (단일 실행) | **Cross-conversation** (세션 간) |
|---|---|---|
| **Static** (불변) | User ID, DB 연결, 도구 정의 | 설정 파일 등 |
| **Dynamic** (가변) | 대화 히스토리, 중간 결과 | 사용자 선호도, 학습된 메모리 |

### 3가지 컨텍스트 타입

| 타입 | Mutability | Lifetime | 예시 | LangGraph 구현 |
|---|---|---|---|---|
| Static Runtime | Static | Single run | User ID, DB conn | `context_schema` |
| Dynamic Runtime (State) | Dynamic | Single run | Messages, 중간결과 | `state_schema` |
| Dynamic Cross-conv (Store) | Dynamic | Cross-conversation | 선호도, 메모리 | `InMemoryStore` |

### 제어 가능한 3가지 컨텍스트 카테고리

| 카테고리 | 제어 대상 | 특성 |
|---|---|---|
| **Model Context** | Instructions, 메시지 히스토리, 도구, 응답 형식 | Transient (일시적) |
| **Tool Context** | 도구 접근, 상태 읽기/쓰기, 런타임 컨텍스트 | Persistent (영구적) |
| **Life-cycle Context** | 단계 간 변환, 요약, 가드레일 | Persistent (영구적) |

LangChain은 **미들웨어(middleware)** 메커니즘으로 컨텍스트 엔지니어링을 구현합니다. `@dynamic_prompt`, `@wrap_model_call` 등의 미들웨어로 컨텍스트를 업데이트하거나 라이프사이클 단계 간 제어를 할 수 있습니다.

## 4.3 정적 런타임 컨텍스트 -- `context_schema` + `@dataclass`

에이전트 실행 중 **변하지 않는** 정보를 `context_schema`로 주입합니다. `@dataclass`로 스키마를 정의하고, 도구에서 `ToolRuntime[Context]`로 접근합니다.

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime
from langchain.agents import create_agent


@dataclass
class UserContext:
    user_id: str
    role: str
    department: str

In [ ]:
@tool
def get_permissions(runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자의 역할에 따른 권한을 조회합니다."""
    ctx = runtime.context
    perms = {"admin": "read,write,delete", "editor": "read,write"}
    return f"사용자 {ctx.user_id} ({ctx.department}): {perms.get(ctx.role, 'read')}"

In [ ]:
agent = create_agent(
    model=model,
    tools=[get_permissions],
    context_schema=UserContext,
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": "제 권한이 어떻게 되나요?"}]},
    context=UserContext(user_id="u42", role="admin", department="eng"),
    config=lf_config,
)
print(result["messages"][-1].content)

### 핵심 포인트

- `context_schema`에 `@dataclass`를 전달하면 타입 안전한 컨텍스트를 사용할 수 있습니다
- 도구 함수에서 `runtime: ToolRuntime[Context]` 타입힌트로 자동 주입됩니다
- 실행 중에는 **읽기 전용**이며 변경되지 않습니다
- 적합한 데이터: User ID, DB 연결, API 키, 세션 메타데이터

## 4.4 동적 런타임 컨텍스트 -- `state_schema`, `AgentState` 커스텀

에이전트가 메시지를 처리하고 도구를 호출하면서 **변화하는** 상태입니다. `AgentState`를 상속하여 커스텀 필드를 추가합니다.

In [ ]:
from langchain.agents import AgentState


class RAGState(AgentState):
    """동적 검색 컨텍스트를 포함한 상태."""

    retrieved_docs: list[str]
    query_count: int


print(f"상태 키: {list(RAGState.__annotations__.keys())}")

In [ ]:
agent_with_state = create_agent(
    model=model,
    tools=[get_permissions],
    state_schema=RAGState,
    context_schema=UserContext,
)
result = agent_with_state.invoke(
    {
        "messages": [{"role": "user", "content": "안녕하세요!"}],
        "retrieved_docs": [],
        "query_count": 0,
    },
    context=UserContext(user_id="u1", role="editor", department="sales"),
    config=lf_config,
)
print(f"최종 상태 키: {list(result.keys())}")

### Static vs Dynamic 비교

| 구분 | Static Runtime (`context_schema`) | Dynamic Runtime (`state_schema`) |
|---|---|---|
| 변경 여부 | 불변 (읽기 전용) | 가변 (노드가 업데이트) |
| 전달 방식 | `context=` 파라미터 | invoke 입력 dict |
| 접근 방법 | `runtime.context.field` | `state["field"]` |
| 적합한 데이터 | 인증 정보, 설정 | 대화 히스토리, 중간 결과 |

## 4.5 장기 메모리 -- InMemoryStore 기본 API

Cross-conversation 컨텍스트를 위해 `InMemoryStore`를 사용합니다. 장기 메모리는 세션과 스레드를 초월하여 지속되는 사용자별 또는 앱 수준의 데이터입니다.

### 저장 구조
메모리는 **JSON 문서** 형태로 저장되며, 계층적 **namespace**로 조직화됩니다:
- **namespace**: 메모리를 분류하는 폴더 역할 (예: `(user_id, "preferences")`)
- **key**: 각 메모리의 고유 식별자 (예: `"theme"`)
- 네임스페이스에는 보통 사용자 ID나 조직 ID를 포함하여 정보 관리를 용이하게 합니다

### 기본 API
| API | 설명 |
|---|---|
| `store.put(namespace, key, value)` | 메모리 저장 (upsert) |
| `store.get(namespace, key)` | 특정 키로 메모리 조회 |
| `store.search(namespace)` | 네임스페이스 내 전체 검색 |
| `store.search(namespace, filter={...})` | 필터 조건으로 검색 |

프로덕션 환경에서는 `InMemoryStore` 대신 **DB 기반 Store** (예: PostgreSQL)를 사용해야 합니다.

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
user_id = "user_42"
store.put((user_id, "preferences"), "theme", {"value": "dark"})
store.put((user_id, "preferences"), "language", {"value": "ko"})

item = store.get((user_id, "preferences"), "theme")
print(f"테마: {item.value}")

In [ ]:
items = store.search((user_id, "preferences"))
for item in items:
    print(f"  [{item.key}] = {item.value}")

filtered = store.search((user_id, "preferences"), filter={"value": "dark"})
print(f"필터 결과: {len(filtered)}건")

## 4.6 장기 메모리 -- 시맨틱 검색

임베딩 함수를 설정하면 `InMemoryStore`가 **시맨틱 검색**을 지원합니다. `query` 파라미터로 의미 기반 유사도 검색을 수행합니다.

In [ ]:
semantic_store = InMemoryStore(index={"embed": embeddings, "dims": 1536})
ns = ("user_42", "memories")
semantic_store.put(ns, "mem1", {"content": "pytest를 unittest보다 선호"})
semantic_store.put(ns, "mem2", {"content": "모든 함수에 타입 힌트 사용"})
semantic_store.put(ns, "mem3", {"content": "좋아하는 음식은 초밥"})
semantic_store.put(ns, "mem4", {"content": "ML 인프라 팀에서 근무"})
print("임베딩과 함께 메모리 4개 저장 완료.")

In [ ]:
results = semantic_store.search(
    ("user_42", "memories"), query="testing preferences", limit=2
)
for r in results:
    print(f"  [{r.key}] {r.value['content']}")

In [ ]:
results2 = semantic_store.search(
    ("user_42", "memories"), query="machine learning work", limit=2
)
for r in results2:
    print(f"  [{r.key}] {r.value['content']}")

### 기본 Store vs 시맨틱 Store 비교

| 기능 | `InMemoryStore()` | `InMemoryStore(index={...})` |
|---|---|---|
| 정확 키 조회 | `get(ns, key)` | `get(ns, key)` |
| 필터 검색 | `search(ns, filter={...})` | `search(ns, filter={...})` |
| 시맨틱 검색 | 불가 | `search(ns, query="...", limit=N)` |
| 프로덕션 | `InMemoryStore` 대신 DB 백엔드 사용 | PostgreSQL 기반 Store 권장 |

## 4.7 도구에서 Store 읽기/쓰기 -- `ToolRuntime.store`

에이전트의 도구 내에서 Store에 접근하여 사용자 정보를 읽고 쓸 수 있습니다. `create_agent(store=...)`로 Store를 연결하면 `runtime.store`로 자동 주입됩니다.

### 읽기 패턴
도구에서 `runtime.store`를 통해 저장된 사용자 정보를 조회합니다. `ToolRuntime[Context]` 타입힌트로 컨텍스트와 Store 모두에 접근할 수 있습니다.

### 쓰기 패턴
도구 파라미터로 사용자 입력을 받아 `store.put()`으로 메모리를 저장합니다. 이를 통해 에이전트가 대화 중 학습한 정보를 영구 저장할 수 있습니다.

### 핵심 사항
- `runtime.store`: Store 인스턴스에 접근
- `runtime.context`: 정적 런타임 컨텍스트에 접근
- Store와 Context를 결합하면 **"누구의(context) 어떤 정보(store)"**를 체계적으로 관리할 수 있습니다

In [ ]:
@tool
def get_user_info(runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자의 저장된 정보를 조회합니다."""
    store = runtime.store
    user_id = runtime.context.user_id
    info = store.get(("users",), user_id)
    return str(info.value) if info else "사용자 정보를 찾을 수 없습니다."

In [ ]:
@tool
def save_preference(key: str, value: str, runtime: ToolRuntime[UserContext]) -> str:
    """사용자 선호도를 저장합니다."""
    store = runtime.store
    user_id = runtime.context.user_id
    store.put((user_id, "preferences"), key, {"value": value})
    return f"선호도 저장됨: {key}={value}"

In [ ]:
memory_agent = create_agent(
    model=model,
    tools=[get_user_info, save_preference],
    context_schema=UserContext,
    store=semantic_store,
)
result = memory_agent.invoke(
    {"messages": [{"role": "user", "content": "테마를 다크로 저장해 주세요"}]},
    context=UserContext(user_id="u42", role="admin", department="eng"),
    config=lf_config,
)
print(result["messages"][-1].content)

## 4.8 메모리 3유형: Semantic, Episodic, Procedural

장기 메모리는 인지과학에서 영감을 받은 세 가지 유형으로 분류됩니다. 각 유형에 따라 **저장 구조**와 **활용 방식**이 다릅니다.

| 유형 | 설명 | 예시 | 구조 |
|---|---|---|---|
| **Semantic** | 엔티티에 대한 사실적 지식 | 사용자 선호도, 프로필 정보 | Profile 또는 Collection |
| **Episodic** | 과거 경험과 이벤트 기억 | Few-shot 예시, 과거 액션 로그 | Collection |
| **Procedural** | 수행 방법에 대한 규칙/지침 | 시스템 프롬프트 수정, 가이드라인 | Profile (규칙 목록) |

### Semantic Memory -- Profile vs Collection

Semantic 메모리는 저장 전략에 따라 두 가지 접근법이 있습니다:

| 접근법 | 구조 | 적합한 경우 | 예시 |
|---|---|---|---|
| **Profile** | 단일 JSON 문서, 지속 업데이트 | 소수의 잘 알려진 속성 | `{"name": "Alice", "language": "Python", "preferred_style": "concise"}` |
| **Collection** | 다수의 좁은 범위 문서, 높은 리콜 | 오픈엔드 또는 대규모 지식 | `[{"topic": "testing", "content": "Prefers pytest"}, ...]` |

### Episodic Memory
과거에 유사한 상황에서 어떻게 행동했는지를 기록합니다. Few-shot 예시로 활용되어 에이전트가 과거 경험에서 학습할 수 있게 합니다.

### Procedural Memory
에이전트의 행동 규칙을 저장합니다. 시스템 프롬프트를 동적으로 수정하는 효과를 가져, 에이전트가 사용자별 맞춤 지침을 따르도록 합니다.

In [ ]:
mem_store = InMemoryStore(index={"embed": embeddings, "dims": 1536})
uid = "user_42"

# Semantic -- Profile (single JSON)
mem_store.put(
    (uid, "profile"),
    "main",
    {
        "name": "Alice",
        "language": "Python",
        "preferred_style": "concise",
    },
)
# Semantic -- Collection (multiple docs)
mem_store.put((uid, "facts"), "f1", {"content": "pytest 선호"})

In [ ]:
# Episodic -- past experiences (few-shot)
mem_store.put(
    (uid, "episodes"),
    "ep1",
    {
        "content": "SQL 최적화 -> EXPLAIN ANALYZE 사용",
    },
)

# Procedural -- rules/guidelines
mem_store.put(
    (uid, "procedures"),
    "rules",
    {
        "content": "항상 에러 처리를 포함. logging 사용.",
    },
)
print("3가지 메모리 유형 모두 저장 완료.")

In [ ]:
# Episodic search: find similar past experiences
episodes = mem_store.search((uid, "episodes"), query="database query help", limit=1)
for ep in episodes:
    print(f"관련 에피소드: {ep.value['content']}")

## 4.9 Progressive Disclosure -- Skills 패턴

모든 컨텍스트를 프롬프트에 넣으면 토큰 비용이 증가하고 정확도가 떨어집니다. Skills 패턴은 **필요할 때만 관련 정보를 로드**하는 Progressive Disclosure 방식입니다.

### Skill의 구조
Skill은 `{name, description, content}`로 구성된 지식 단위입니다:
- **name**: 스킬 식별자 (예: `"customers_schema"`)
- **description**: 짧은 설명 (시스템 프롬프트에 포함됨)
- **content**: 상세 내용 (`load_skill` 도구로 온디맨드 로드)

### 크기별 전략

| 크기 | 전략 | 예시 |
|---|---|---|
| **<1K tokens** | 시스템 프롬프트에 직접 포함 | 테이블 이름, 고수준 관계 |
| **1-10K tokens** | `load_skill` 도구로 온디맨드 로드 | 테이블 스키마, 쿼리 패턴, 베스트 프랙티스 |
| **>10K tokens** | 페이지네이션으로 온디맨드 로드 | 대규모 참조 데이터, 과거 쿼리 로그 |

### 동작 흐름
1. **미들웨어**가 모든 스킬의 이름과 설명을 시스템 프롬프트에 주입
2. 에이전트가 질문을 분석하고 필요한 스킬을 판단
3. `load_skill` 도구를 호출하여 상세 내용을 로드
4. 로드된 내용을 바탕으로 작업 수행

### 장점

| 장점 | 설명 |
|---|---|
| **토큰 효율성** | 현재 쿼리에 필요한 정보만 로드 |
| **확장성** | 수백 개의 테이블이 있는 DB도 지원 |
| **정확도** | 필요한 시점에 상세 스키마를 제공 |
| **비용 절감** | 요청당 입력 토큰 감소 |

In [ ]:
skills = [
    {
        "name": "db_overview",
        "description": "모든 테이블의 고수준 개요",
        "content": "테이블: customers, orders, products",
    },
    {
        "name": "customers_schema",
        "description": "customers 테이블의 전체 스키마",
        "content": "CREATE TABLE customers (id INT PK, name VARCHAR)",
    },
]
SKILL_MAP = {s["name"]: s for s in skills}
print(f"스킬 {len(skills)}개 정의됨.")

In [ ]:
from langchain_core.tools import tool


@tool
def load_skill(skill_name: str) -> str:
    """데이터베이스 스킬에 대한 상세 정보를 로드합니다."""
    skill = SKILL_MAP.get(skill_name)
    if skill is None:
        return f"찾을 수 없음. 사용 가능: {', '.join(SKILL_MAP.keys())}"
    return f"## {skill['name']}\n\n{skill['content']}"

## 4.10 Hot Path vs Background 메모리 쓰기

메모리를 **언제** 쓰느냐에 따라 사용자 응답 지연에 영향을 미칩니다.

| 방식 | 타이밍 | 즉시 사용 가능? | 지연 영향 |
|---|---|---|---|
| **Hot path** | 대화 루프 내 실시간 | 즉시 (다음 턴에 사용 가능) | 응답 지연 증가 |
| **Background** | 별도 비동기 태스크 | 지연됨 (Eventual Consistency) | 지연 영향 없음 |

### Hot Path 쓰기
에이전트 루프 내 인라인으로 메모리를 저장합니다. 바로 다음 턴에서 해당 메모리를 사용해야 할 때 적합합니다. 예: 사용자가 방금 알려준 선호도를 즉시 반영해야 하는 경우.

### Background 쓰기
별도의 프로세스나 비동기 태스크로 메모리를 저장합니다. Eventual Consistency가 허용되는 경우에 사용하며, 응답 지연에 영향을 주지 않습니다. 예: 대화 패턴 분석, 장기 학습 데이터 축적.

### 선택 기준
- 즉시 리콜이 필요한가? -> **Hot path**
- 지연 감소가 우선인가? -> **Background**
- 대부분의 경우 Background 쓰기를 선호합니다

In [ ]:
from langgraph.store.base import BaseStore


# Hot path: write inline (adds latency)
def reflect_node(state, store: BaseStore):
    """메모리를 인라인으로 추출하고 저장합니다."""
    last_msg = state["messages"][-1].content
    store.put(("user", "reflections"), "latest", {"content": last_msg})
    return state


print("Hot path: 즉시 저장, 다음 턴에 사용 가능.")

In [ ]:
import asyncio


# Background: write in separate async task
async def background_memory_writer(state, store: BaseStore):
    """백그라운드에서 메모리를 저장합니다 (지연 없음)."""
    last_msg = state["messages"][-1].content
    await store.aput(("user", "reflections"), "latest", {"content": last_msg})


print("Background: 최종 일관성, 지연 없음.")

## 요약

### 컨텍스트 엔지니어링 3요소

| 요소 | 구현 | API |
|---|---|---|
| 정적 런타임 | `context_schema` + `@dataclass` | `runtime.context.field` |
| 동적 런타임 | `state_schema` + `AgentState` | `state["field"]` |
| 장기 메모리 | `InMemoryStore` + `store=` | `store.put/get/search` |

### 메모리 3유형

| 유형 | 용도 | Namespace 예시 |
|---|---|---|
| Semantic | 사용자 프로필/사실 | `(user_id, "profile")`, `(user_id, "facts")` |
| Episodic | 과거 경험 (few-shot) | `(user_id, "episodes")` |
| Procedural | 규칙/프롬프트 수정 | `(user_id, "procedures")` |

### Best Practices

| 원칙 | 설명 |
|---|---|
| 정적 컨텍스트 최소화 | 현재 태스크에 필요한 것만 포함 |
| Namespace 구조화 | 충돌 방지를 위해 계층적 namespace 사용 |
| 시맨틱 검색 우선 | 정확 매칭보다 임베딩 기반 검색이 확장성 우수 |
| Background 쓰기 선호 | 즉시 리콜 불필요시 background로 지연 감소 |
| Skills 패턴 적용 | 대규모 컨텍스트는 Progressive Disclosure |

### 다음 단계
→ **[05_agentic_rag.ipynb](./05_agentic_rag.ipynb)**: Agentic RAG를 배웁니다.
